In [6]:
from pyspark.sql import SparkSession
from pyspark.mllib.tree import RandomForest
from pyspark.mllib.regression import LabeledPoint
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.mllib.linalg import Vectors
from pyspark.sql.functions import when
import numpy as np


In [7]:
spark = SparkSession.builder.appName("RandomForerst").getOrCreate()
data = spark.read.csv("data.csv", header = True, inferSchema=True)

In [8]:
# M is encoded as 1.0 and B is encoded as 0.0
data = data.withColumn("label", 
                              when(data.diagnosis == "M", 1.0)
                              .otherwise(0.0))


feature_cols = [col for col in data.columns if col not in ['id', 'diagnosis', 'label']]
model_data = data.select(feature_cols + ['label'])

def create_labeled_point(row):
    features = [row[col] for col in feature_cols]
    return LabeledPoint(row.label, Vectors.dense(features))

rdd_data = model_data.rdd.map(create_labeled_point)

In [9]:
train_rdd, test_rdd = rdd_data.randomSplit([0.7, 0.3], seed=42)

In [ ]:
rf = RandomForest.trainClassifier(data=train_rdd,
                                  numClasses=2,
                                  categoricalFeaturesInfo={},
                                  numTrees=10,
                                  impurity='gini',
                                  maxDepth=3,
                                  seed=42)
predictions = rf.predict(test_rdd.map(lambda x: x.features))


Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 1 times, most recent failure: Lost task 0.0 in stage 5.0 (TID 5) (DESKTOP-E6MA3J1 executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "C:\Users\Jason\anaconda3\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 1100, in main
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 10) than that in driver 3.12, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.s

In [97]:
predictions = model.transform(df_test)

In [98]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="prediction", 
    metricName="accuracy"
)
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="prediction", 
    metricName="f1"
)
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="prediction", 
    metricName="weightedRecall"
)
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", 
    predictionCol="prediction", 
    metricName="weightedPrecision"
)

accuracy = accuracy_evaluator.evaluate(predictions)
f1_score = f1_evaluator.evaluate(predictions)
recall = recall_evaluator.evaluate(predictions)
precision = precision_evaluator.evaluate(predictions)

print(f"Model Performance:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1 Score:  {f1_score:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

Model Performance:
Accuracy:  0.9563
F1 Score:  0.9562
Precision: 0.9562
Recall:    0.9563


In [99]:
df_test.count()

160

In [ ]:
# Should total up to equal df_test.count
confusion_matrix_df = predictions.groupBy("label", "prediction").count().orderBy("label", "prediction")
confusion_matrix_df.show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|   99|
|  0.0|       1.0|    3|
|  1.0|       0.0|    4|
|  1.0|       1.0|   54|
+-----+----------+-----+

